# chrisMain — `EyeMovementTrajectoryAlternatingBackground`

Workflow notebook **tuned for the `EyeMovementTrajectoryAlternatingBackground`
protocol** (short name: `eye_movement_alt_bg`). All defaults — QC
thresholds, condition keys, movie-repeat cycle length, population
comparison axis — assume this protocol. **For a different protocol,
copy this notebook and adapt §4 thresholds + §11 analyzer choices;
do not edit chrisMain in place.**

## Sections at a glance

- **Setup (§1–§3)** — list protocol datafiles, pick one, build the
  `(StimBlock, ResponseBlock, AnalysisChunk)` pipeline.
- **QC + archive (§4 → §5 → §6/§9)** — automated protocol QC →
  optional click-through visual QC → per-cell PNG archive (single
  date or batch over many dates).
- **Spike-sorting QC (§7, §8)** — confirm spikes are assigned to
  the right cell. §7 writes static PNGs (good for batch / remote
  review); §8 is the interactive ipywidgets GUI (windowed loading,
  bandwidth meter — designed for remote NAS sessions).
- **Offline store (§10)** — pack QC-good cells into a single HDF5
  per date so subsequent sessions skip DataJoint + the SSD pipeline.
- **Analyses (§11–§12)** — protocol-specific offline analyses
  (`retinanalysis.protocols.eye_movement_alt_bg`) per date, then
  cross-date pooling.

## Workflow order (first time through)

1. Run §1 → §3 to build the pipeline for a single date.
2. Run §4 to compute `qc.csv` (auto firing-rate + silent-epoch gates).
3. Run §6 (or §9 for batch) to render the per-cell PNG archive.
4. Run §5 to tag cells `good` / `bad` interactively → `visual_qc.csv`.
5. Re-run §6 / §9: the archive now prunes to the curated set.
6. (Optional) Run §7 or §8 to sanity-check the sort itself.
7. Run §10 to write `offline.h5`, then §11 / §12 for analyses.

Sections marked **(optional)** can be skipped on a first pass.

## Reference

- Repo conventions: `CLAUDE.md` at the repo root.
- Removed exploratory cells (single-cell STA/EI inspection, regen
  stimulus + canvas overlay, raster/PSTH spot checks, manual rig
  calibration, EI-match diagnostics) are recoverable from git history:
  `git show <pre-consolidation-sha>:demos/chrisMain.ipynb`.

In [1]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

## 1. Find all experiments that ran `EyeMovementTrajectoryAlternatingBackground`

`ra.get_datasets_from_protocol_names()` does a **lowercase substring
match** against the protocol registry, so the short query
`'AlternatingBackground'` catches the full Java class name
(`edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground`)
without you typing it. The result has **one row per (exp_name,
datafile_name)** so multiple datafiles of the same protocol on one date
are already separated.

We then filter to experiments whose Kilosort output is actually on
disk under `ra.ANALYSIS_DIR` — registry rows without local sort data
are dropped silently.

In [2]:
exp_search = ra.get_datasets_from_protocol_names('AlternatingBackground')

# Filter to experiments that exist on the MEA SSD
available_experiments = os.listdir(ra.ANALYSIS_DIR)
exp_search = exp_search.query('exp_name in @available_experiments').reset_index(drop=True)

print(f'{len(exp_search)} usable datafile(s) found across '
      f'{exp_search.exp_name.nunique()} experiment(s).')
display(exp_search)


Found 1 protocols matching "alternatingbackground":
['edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground']

Found 22 experiments, 31 epoch blocks.

26 usable datafile(s) found across 19 experiment(s).


,exp_name,datafile_name,NDF,chunk_name,protocol_name,is_mea,data_dir,group_label,experiment_id,protocol_id,group_id,block_id,chunk_id
0,20230214C,data018,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230214C/data018,eye movement alt background ndf 3.0,39,41,781,1484,97
1,20230313C,data017,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data017,eye movement alt background ndf 3.0,42,41,878,1609,108
2,20230313C,data019,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data019,Var Mean Drift Grating ndf 3.0,42,41,879,1611,108
3,20230502C,data018,2.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230502C/data018,ndf 2.0 eye movement alt background,52,41,1077,1855,156
4,20230523C,data013,2.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230523C/data013,Eye movement alt background,56,41,1178,1983,176
5,20230525C,data052,0.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230525C/data052,Eye Movement Alternating Background Photopic,58,41,1220,2041,184
6,20230725C,data036,2.0,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230725C/data036,ndf 2 eye movement alt background,65,41,1397,2264,205
7,20231003C,data021,0.5,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231003C/data021,eye movement alt background,71,41,1516,2416,221
8,20231108C,data005,0.5,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231108C/data005,eye movement alt background,74,41,1556,2462,243
9,20231220C,data022,0.5,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20231220C/data022,CC eye movement alt background,76,41,1607,2517,252


## 2. Pick a (date, datafile)

Edit `ENTRY_INDEX` below to a row index from §1's table. The notebook
binds `exp_name` and `datafile_name` together from that one row, so
they can't drift apart. To switch dates, just change the index and
re-run from §3 down.

The cell also displays a short summary of all datafiles recorded on
the chosen date (capped at 10 rows) for context — useful when checking
that you picked the right run of `EyeMovementTrajectoryAlternatingBackground`
rather than an adjacent protocol.

In [ ]:
ENTRY_INDEX = 14   # <-- EDIT ME: row index in exp_search above

_entry = exp_search.iloc[ENTRY_INDEX]
exp_name      = _entry['exp_name']
datafile_name = _entry['datafile_name']
print(f'Selected entry {ENTRY_INDEX}: {exp_name} / {datafile_name}')

# Summary of the chosen day, capped at SUMMARY_HEAD_N rows so long experiments
# don't blow up the notebook output. Increase the cap if you need to inspect
# more rows (or call ra.get_exp_summary(exp_name) directly to see all of them).
SUMMARY_HEAD_N = 10
_sum_df = ra.get_exp_summary(exp_name)
print(f'\nexperiment summary: {len(_sum_df)} datafiles total '
      f'(showing first {min(SUMMARY_HEAD_N, len(_sum_df))})')
display(_sum_df.head(SUMMARY_HEAD_N))


## 3. Mirror Vision files to local cache (optional), then build the pipeline

Building the pipeline pulls every Vision file (`.ei`, `.neurons`,
`.params`, `.classification.txt`, ...) for one protocol datafile and
one noise chunk — ~1 GB total. When `find_path` resolves to a remote
NAS, every kernel restart pays that bandwidth bill again.

### Step 3a — mirror to local disk (run once per experiment)

`ra.mirror_to_local_cache(exp_name, datafile_name, chunk_name)` copies
the needed Vision files into `~/.cache/retinanalysis/` (override the
path via the `RA_LOCAL_CACHE_ROOT` environment variable). The
`local_cache` tier sits at the top of `find_path`'s priority list, so
**every subsequent pipeline build reads from local disk** — no NAS
traffic at all. The huge `.sta` raw STA movie is skipped by default
(the rest of the pipeline only needs STA *fits* from `.params`); pass
`include_sta=True` to bring it too.

Run-once flow:
1. First time: pays the ~1 GB transfer.
2. Subsequent kernel restarts: `ra.mirror_to_local_cache(...)` returns
   instantly with "cached" lines for every file (size + mtime match).

`ra.local_cache_status()` shows what is currently cached;
`ra.clear_local_cache(exp_name)` reclaims disk space when done.

**Skip this step** when you're working off a local SSD already — the
mirror is harmless but wastes time copying files to a different local
disk. The cache check defers to `find_path`'s tier priority either
way.

### Step 3b — `ra.create_mea_pipeline()`

Constructs everything downstream cells need in one call:

- **`stim_block`** — `MEAStimBlock`: stimulus parameters and frame
  timing for the protocol datafile. For
  `EyeMovementTrajectoryAlternatingBackground` this carries
  `preTime`/`stimTime`/`tailTime`, `currentImageName`,
  `currentBackgroundScale`, and the eye-movement trajectory.
- **`response_block`** — `MEAResponseBlock`: spike times pulled from
  the Kilosort output for *this* datafile (one list per cell × epoch).
- **`analysis_chunk`** — the noise chunk: STAs, RF params, EIs, ISIs,
  timecourses, classification. Auto-resolved to the chunk closest in
  time to the protocol datafile; override by setting
  `noise_chunk_name = "chunkN"` at the top of the cell.
- **EI cluster-match** — runs `vision_utils.cluster_match()` to align
  noise-chunk cell IDs to protocol cell IDs via EI footprint
  correlation. After this, `response_block.df_spike_times` carries
  `noise_id` and `cell_type` columns.

### Knobs you usually leave alone

`ei_corr_cutoff=0.65`, `ei_match_method='all'`, `ei_use_isi=False`,
`ei_use_timecourse=False`, `ei_n_removed_channels=1`. Loosen
`ei_corr_cutoff` (e.g. 0.5) if matches drop on a noisy date; raise it
(e.g. 0.8) if you suspect spurious matches.

### Pitfall

Older experiments are often sorted with `kilosort2` (not `kilosort2.5`)
— the cell auto-detects available versions. If both exist, `kilosort2.5`
wins. macOS AppleDouble dotfiles (`._kilosort*.classification.txt`)
are explicitly skipped.

In [ ]:
# Step 3a — mirror this date's Vision files to the local cache.
# Safe to re-run: files already present are skipped (size + mtime match).
# Skip / comment out this cell when working off a local SSD already.
import os

# Auto-detect ss_version + noise chunk for the mirror so the user
# doesn't have to set them twice (§3b will resolve them again the
# same way). For the noise chunk we use the same MEAStimBlock auto-pick
# as the pipeline builder below — the auto-pick is cheap (a small DJ
# query) and doesn't touch NAS Vision files.
_sort_dir = os.path.join(ra.DATA_DIR, exp_name, datafile_name)
_available_ss = [d for d in os.listdir(_sort_dir) if d.startswith('kilosort')]
_ss_version = 'kilosort2.5' if 'kilosort2.5' in _available_ss else _available_ss[0]

# Pin a noise chunk if you want; otherwise we resolve the auto-pick once.
from retinanalysis.classes.stim import MEAStimBlock
_tmp = MEAStimBlock(exp_name, datafile_name, verbose=False)
_chunk = _tmp.nearest_noise_chunk

print(f'Mirror plan:  {exp_name} / {datafile_name} (ss={_ss_version}) + chunk {_chunk}\n')

mirror_report = ra.mirror_to_local_cache(
    exp_name,
    datafile_name=datafile_name,
    chunk_name=_chunk,
    ss_version=_ss_version,
    include_sta=False,    # raw .sta is ~9 GB and not needed by the pipeline
    verbose=True,
)
print(f'\nLocal cache root: {ra.LOCAL_CACHE_ROOT}')
print(f'  copied this run: {mirror_report["bytes_copied_total"] / 1e6:.1f} MB')
print(f'  total on disk  : {mirror_report["bytes_total"] / 1e6:.1f} MB')


In [ ]:
# Older experiments are often sorted with kilosort2 (not 2.5). Auto-detect:
exp_sort_dir = os.path.join(ra.DATA_DIR, exp_name, datafile_name)
available_ss = [d for d in os.listdir(exp_sort_dir) if d.startswith('kilosort')]
ss_version = 'kilosort2.5' if 'kilosort2.5' in available_ss else available_ss[0]

# ---- USER INPUT --------------------------------------------------------
# Noise chunk to use for cell typing + EI matching.
#   None  → MEAStimBlock.nearest_noise_chunk (closest in time, either direction)
#   'chunkN' → pin explicitly. Bypasses get_nearest_noise.
noise_chunk_name = None
typing_file_name = None           # e.g. 'kilosort2.5.classification.txt', or None for first match

# EI cluster-match knobs (forwarded to vision_utils.cluster_match).
ei_corr_cutoff       = 0.65        # minimum max-correlation to accept a match (0.6 looser, 0.9 stricter)
ei_match_method      = 'all'      # 'all' (max of three), 'full', 'space', 'power'
ei_use_isi           = False      # also require ISI corr ≥ 0.3
ei_use_timecourse    = False      # also require RGB timecourse corr ≥ 0.3
ei_n_removed_channels = 1         # drop this many top-amplitude electrodes per EI before correlating
# ------------------------------------------------------------------------

# DB record: each datafile row carries a chunk_name written at ingest time.
# That's what the experimenter declared at the rig. Use it as a sanity check
# against whatever auto-pick or user-pin we end up using below.
_exp_df = ra.get_exp_summary(exp_name)
_db_chunk_row = _exp_df.query('datafile_name == @datafile_name')
db_chunk_name = None
if not _db_chunk_row.empty:
    db_chunk_name = str(_db_chunk_row['chunk_name'].iloc[0])
    print(f'database chunk_name for {datafile_name}: {db_chunk_name}')

# Resolve noise chunk: explicit override > auto-pick via MEAStimBlock.
# When pinned, MEAStimBlock will skip get_nearest_noise() inside the
# pipeline build below — no need to construct a throwaway block here.
if noise_chunk_name is None:
    from retinanalysis.classes.stim import MEAStimBlock
    _tmp_stim = MEAStimBlock(exp_name, datafile_name, verbose=False)
    noise_chunk_name = _tmp_stim.nearest_noise_chunk
    print(f'auto-picked noise chunk: {noise_chunk_name}')
else:
    print(f'using user-specified noise chunk: {noise_chunk_name}')

# Sanity check: does the chunk we will load match the one the experimenter
# declared in the DB? Mismatch is allowed (the auto-pick or the user might
# correctly want a different chunk) but worth flagging loudly.
if db_chunk_name is not None and db_chunk_name != noise_chunk_name:
    print(f'\n*** ALERT: DB chunk {db_chunk_name!r} differs from chunk being used '
          f'({noise_chunk_name!r}). ***')
    print(f'    If the DB record is correct, set noise_chunk_name = {db_chunk_name!r} '
          f'above and re-run this cell.\n')

# List candidate classification files (skip macOS AppleDouble dotfiles).
chunk_dir = os.path.join(ra.ANALYSIS_DIR, exp_name, noise_chunk_name, ss_version)
typing_candidates = [
    f for f in os.listdir(chunk_dir)
    if f.endswith('.classification.txt') and not f.startswith('.')
]
if not typing_candidates:
    raise FileNotFoundError(
        f'No .classification.txt in {chunk_dir}. '
        f'Pick a different noise_chunk_name above.'
    )
if typing_file_name is None:
    typing_file_name = typing_candidates[0]
elif typing_file_name not in typing_candidates:
    raise FileNotFoundError(
        f'{typing_file_name!r} not found in {chunk_dir}. '
        f'Available: {typing_candidates}'
    )

print(f'ss_version: {ss_version}')
print(f'noise chunk: {noise_chunk_name}')
print(f'typing file: {typing_file_name}')
print(f'(other classification files in chunk: {typing_candidates})')
print(f'EI match: method={ei_match_method!r} cutoff={ei_corr_cutoff} '
      f'use_isi={ei_use_isi} use_timecourse={ei_use_timecourse} '
      f'n_removed_channels={ei_n_removed_channels}')

pipeline = ra.create_mea_pipeline(
    exp_name,
    datafile_name,
    ss_version=ss_version,
    typing_file=typing_file_name,
    analysis_chunk_name=noise_chunk_name,   # honor the override above
    ei_corr_cutoff=ei_corr_cutoff,
    ei_match_method=ei_match_method,
    ei_use_isi=ei_use_isi,
    ei_use_timecourse=ei_use_timecourse,
    ei_n_removed_channels=ei_n_removed_channels,
)

stim_block      = pipeline.stim
response_block  = pipeline.resp
analysis_chunk  = pipeline.analysis_chunk

print(f'\nNoise chunk used: {analysis_chunk.chunk_name}')
print(f'Cells in noise chunk: {len(analysis_chunk.cell_ids)}')
print(f'Cells in protocol datafile: {len(response_block.cell_ids)}')
print(f'Cells matched by EI: {len(pipeline.match_dict)}')
print(f'ei_match_config: {pipeline.ei_match_config}')


database chunk_name for data018: dynamics2
using user-specified noise chunk: dynamics2


FileNotFoundError: No .classification.txt in /Volumes/data/data/sorted/20240523C/dynamics2/kilosort2.5. Pick a different noise_chunk_name above.

## 4. Per-cell QC inside the protocol (`EyeMovementTrajectoryAlternatingBackground`)

A cell can pass classification on the noise chunk and still misbehave
inside a long protocol like this one — drift off, drop out for runs of
trials, or barely fire. `protocol_qc.block_qc_metrics()` returns a
per-cell metrics DataFrame; `filter_cells_by_qc()` adds a boolean
`passes` column.

**Output**: `<OUTPUT_DIR>/<exp>/<protocol_subdir>/qc.csv` — the **initial
good/bad tagging** for every cell. §5 (visual QC), §6/§9 (archives)
and §10 (offline store) all honor it.

### Two automated gates do most of the work

- **Adaptive firing rate.** A cell passes when ≥ `min_frac_epochs_above_rate`
  (default **80%**) of its epochs hit the rate threshold
  (`min_rate_hz × epoch_duration_s`, default 1 Hz). Scales with epoch
  length so the same defaults work across 5 s, 30 s, and 60 s
  protocols (typical for this protocol is ~60 s).
- **Silent-epoch survival.** A cell passes when ≥
  `min_frac_non_silent_epochs` (default **2/3**) of its epochs have
  ≥1 spike — "drop the silent epochs and keep the cell if at least
  two-thirds of its trials survive" without actually dropping epochs.

### Tuning

`OVERWRITE_QC = False` by default: a prior `qc.csv` is loaded as-is.
Flip to `True` after editing `MIN_RATE_HZ` / `MIN_FRAC_EPOCHS` /
`MIN_FRAC_NON_SILENT` to recompute and overwrite. The summary printed
at the bottom reports pass rate per cell type and the first few failing
cells with their gate scores — quick way to see whether the gates are
biased against a specific type.

### Other reportable metrics (off by default; gate by setting thresholds)

`min_count_per_epoch`, `cv_count`, `fano`, `silent_trial_frac`,
`silent_run_max`, `drift_score`, `reliability_r` (split-half PSTH; off
by default for mixed-condition protocols like this one).

In [ ]:
import pandas as pd
from retinanalysis.utils.protocol_qc import (
    block_qc_metrics, filter_cells_by_qc, QCThresholds,
    protocol_qc_csv_path,
)

# ---- USER INPUT --------------------------------------------------------
# When a qc.csv from a prior run already exists for this date, default to
# LOADING it instead of recomputing. Flip OVERWRITE_QC=True to recompute
# even when one exists (e.g. after tweaking the thresholds below).
OVERWRITE_QC = False
# ------------------------------------------------------------------------

# Two adaptive gates — both are user-tunable here.
MIN_RATE_HZ = 1.0                  # firing-rate floor in spikes/s
MIN_FRAC_EPOCHS = 0.8              # fraction of epochs that must meet that rate
MIN_FRAC_NON_SILENT = 2.0 / 3.0    # cell kept iff ≥ this fraction of epochs has ≥1 spike

# Resolve the same protocol subdir §6 will use, so loading matches saving.
from retinanalysis.utils.cell_plot_archive import protocol_short_name
_short = protocol_short_name(response_block.protocol_name)
if 'protocol_subdir' in dir() and protocol_subdir is not None:
    _short = protocol_subdir
elif 'append_datafile_to_subdir' in dir() and append_datafile_to_subdir:
    _short = f'{_short}_{datafile_name}'
_qc_path = protocol_qc_csv_path(exp_name, _short)

if _qc_path.exists() and not OVERWRITE_QC:
    qc = pd.read_csv(_qc_path)
    print(f'Loaded qc.csv from {_qc_path}  ({len(qc)} cells)')
    print('Set OVERWRITE_QC=True above to recompute with current thresholds.')
else:
    if _qc_path.exists():
        print(f'OVERWRITE_QC=True: recomputing and overwriting {_qc_path}')
    else:
        print(f'No qc.csv on disk yet — computing fresh.')

    t_total_ms = (
        response_block.d_timing['pre_time_ms']
        + response_block.d_timing['stim_time_ms']
        + response_block.d_timing['tail_time_ms']
    )
    qc_metrics = block_qc_metrics(
        response_block, t_start_ms=0, t_end_ms=t_total_ms,
        min_rate_hz=MIN_RATE_HZ,
    )
    qc = filter_cells_by_qc(qc_metrics, thresholds=QCThresholds(
        min_rate_hz=MIN_RATE_HZ,
        min_frac_epochs_above_rate=MIN_FRAC_EPOCHS,
        min_frac_non_silent_epochs=MIN_FRAC_NON_SILENT,
    ))
    # Persist this date's QC outcome — initial good/bad tag set for every cell.
    qc_path = ra.save_protocol_qc(qc, exp_name, protocol=_short)
    print(f'qc.csv → {qc_path}')

epoch_s = qc['epoch_duration_s'].iloc[0]
print(f'\nepoch window: {epoch_s:.1f} s')
print(f'  rate gate:        ≥ {MIN_RATE_HZ:.1f} Hz × {epoch_s:.1f} s = '
      f'{MIN_RATE_HZ*epoch_s:.0f} spikes/epoch in ≥{100*MIN_FRAC_EPOCHS:.0f}% of epochs')
print(f'  silent-epoch gate: ≥ {100*MIN_FRAC_NON_SILENT:.0f}% of epochs have ≥1 spike')

n_pass = int(qc.passes.sum())
print(f'\nTotal cells: {len(qc)},  Passing QC: {n_pass} ({100*n_pass/len(qc):.1f}%)')
print('\nPass rate by cell type:')
for ct, sub in qc.groupby('cell_type'):
    rate = sub['mean_rate_hz'].median()
    print(f'  {ct:<12}  {sub.passes.sum():>4} / {len(sub):>4}  '
          f'({100*sub.passes.mean():3.0f}%)   median rate: {rate:5.1f} Hz')

# Show the most informative failures: those that survive the rate gate
# but fail the silent-epoch gate (or vice versa) tell you which gate did
# the work for a given cell.
fails = qc[~qc.passes].sort_values('frac_non_silent_epochs')
print(f'\nFirst few failing cells ({len(fails)} total):')
display(fails[['cell_id', 'cell_type', 'n_epochs', 'mean_rate_hz',
               'frac_epochs_above_rate', 'frac_non_silent_epochs',
               'silent_run_max', 'drift_score']].head().round(2))


## 5. Visual QC (optional) — click through each cell, tag good/bad

**This step is optional.** §4 wrote an automated QC pass/fail to
`qc.csv`. Use this section when you want to **further restrict** the
archive by eyeballing each cell's raster + PSTH.

### Iterative workflow

1. First time through, **skip this section** (no PNGs exist yet) and
   run §6 / §9 to build the initial archive.
2. Come back here once PNGs are on disk — `ra.browse_cells_qc(exp_name)`
   opens an ipywidgets panel that pages through cells (raster left,
   PSTH right) with `Good` / `Bad` / `Prev` / `Next` buttons. Each
   click upserts a row in
   `<OUTPUT_DIR>/<exp>/<protocol_subdir>/visual_qc.csv` — the session
   is resumable.
3. Re-run §6 / §9. They auto-detect `visual_qc.csv` and restrict the
   per-cell PNG render to cells tagged `good`. `cell_match.csv` is
   left comprehensive so downstream EI joins still see the full
   population.

### Downstream selection (no change needed)

```python
cells = ra.select_good_cells()   # uses visual_qc.csv if present, else QC-pass set
```

### Invariant

`visual_qc.csv` is **written only by this GUI**. `analyze_experiment`,
`save_per_cell_plots`, `save_cell_match`, and `save_protocol_qc` are
all read-only with respect to it (audited in
`tests/test_visual_qc_invariant.py`).

Requirements: `ipywidgets` (already in the `retinanalysis` kernel).

In [25]:
# Launch the per-cell GUI for the date picked in §2. If no PNGs exist
# yet, the widget prints a message and returns — run §6 (single date) or
# §9 (batch) first to build the archive, then come back here to tag.
ra.browse_cells_qc(exp_name)

## 6. Archive the picked date (single date)

Run `ra.analyze_experiment(exp_name, datafile_name)` to write the
**full per-cell PNG archive** for the date selected in §2. Output goes
to `<OUTPUT_DIR>/<exp>/<protocol_subdir>/`:

| file | what it is |
|---|---|
| `mosaic.png` | composite: STA mosaic + temporal-filter + ISI rows |
| `index.csv` | one row per archived cell (`cell_id`, `cell_type`, …) |
| `cell_match.csv` | EI-match diagnostics per cell — kept comprehensive |
| `cells/<celltype>/cell_<id>_raster.png` | per-cell raster, condition-colored |
| `cells/<celltype>/cell_<id>_psth.png` | per-cell PSTH, condition-colored |

### Visual-QC integration is automatic

If `visual_qc.csv` exists for this experiment (from §5), the call
below **restricts the per-cell PNG step to cells tagged `good`** — no
extra flags needed. Otherwise it falls back to every cell that passed
§4's automated QC.

### Re-archiving prunes stale PNGs

`prune_stale=True` (default): any `cells/.../cell_<id>_*.png` whose
`cell_id` is *not* in the kept set (QC-pass ∩ visual-QC `good`) is
deleted on re-run. So tagging cells `bad` in §5 and re-running this
cell removes their PNGs from disk. Non-canonical files (e.g. a
`README`) are not touched.

### Pitfall: same-protocol-twice-in-one-day

If a date has **two datafiles of the same protocol**, the default
`protocol_subdir` (the short name, e.g. `eye_movement_alt_bg`) is the
same for both runs — the second would overwrite the first. Set
`append_datafile_to_subdir=True` (or pass an explicit
`protocol_subdir`) to disambiguate.

### Conditions used for raster + PSTH coloring

Auto-detected from this protocol's registry entry; for
`EyeMovementTrajectoryAlternatingBackground` it is
`currentBackgroundScale` (the low/high background pairing).

In [ ]:
# Full archive for the date picked in §2. analyze_experiment now reads
# visual_qc.csv on its own (respect_visual_qc=True by default) and
# restricts the per-cell PNG step to cells tagged 'good' when the file
# exists — no extra notebook glue needed. overwrite=True regenerates
# every targeted PNG.

# ---- USER INPUT --------------------------------------------------------
# Subdirectory under <OUTPUT_DIR>/<exp_name>/. Default uses the protocol
# short name (e.g. 'eye_movement_alt_bg') — fine when the date has one
# datafile of this protocol. When multiple datafiles of the SAME protocol
# exist on this date, set protocol_subdir below (or set
# append_datafile_to_subdir=True) to disambiguate, otherwise each run
# overwrites the previous one.
protocol_subdir = None             # e.g. 'eye_movement_alt_bg_data032', or None
append_datafile_to_subdir = False  # True → auto-append datafile_name
# ------------------------------------------------------------------------

result = ra.analyze_experiment(
    exp_name,
    datafile_name=datafile_name,
    overwrite=True,
    fit_calibration=False,
    n_jobs=-1,
    verbose=True,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
)
print(f'\nDone: {result["exp_name"]} / {result["datafile_name"]} — '
      f'QC-pass pool: {result["n_cells_passed_qc"]}/{result["n_cells_total"]}')
print(f'  output_dir: {result["output_dir"]}')


## 7. Spike-sorting QC — static PNGs (batch / remote review)

PSTH + raster confirm that spike *times* are consistent with the
stimulus; they don't tell you whether **the spikes were assigned to
the right cell** in the first place. This cell samples a few cells
(QC-pass ∩ visual-QC `good`) per type and writes one **multi-row PNG
per cell**: each row is one full epoch (raster strip on top, 300-Hz
high-pass-filtered raw trace below, red dots at the cell's spike
times, snapped to the local trough in ±2 ms).

A clean sort: red dots land on visible spike waveforms in the trace.
A merge: extra waveforms in the trace with no red dot, *or* red dots
on flat baseline (template hits that aren't real spikes).

### Output

`<OUTPUT_DIR>/<exp>/<protocol_subdir>/sorting_qc_<protocol_short>_<datafile>/cell_proto<XXXX>_noise<YYYY>_<celltype>_sorting_qc.png`

The folder name stamps both the protocol short name and the datafile
so multiple runs of `EyeMovementTrajectoryAlternatingBackground` on
one date don't collide.

### When to prefer this over §8

- **Batch review of many cells / dates**: PNGs are reviewable offline
  and shareable.
- **Slow or metered connection**: a 4-epoch sample is heavy
  (~3.6 GB read from raw `.bin` per cell on a typical 60-s epoch). §8
  loads only a sub-window per click.

### When to prefer §8

- **Spot checks** of one cell at a time with interactive zoom.
- **Remote NAS sessions** — see the bandwidth chip in §8.

In [ ]:
# §7 — Sorting QC via raw traces, saved to disk as high-DPI PNGs.
# Samples from QC-pass ∩ visual-QC 'good' cells per type and writes one PNG
# per cell to <OUTPUT>/<exp>/<protocol_subdir>/sorting_qc_<protocol>_<datafile>/.
# Each PNG has N_EPOCHS full-width rows; every row = thin raster strip +
# 300 Hz HP-filtered trace with red marks at the cell's spike times.

CELL_TYPES         = ['OnP', 'OnM']    # which types to sample
N_CELLS_PER_TYPE   = 3                  # cells per type
N_EPOCHS           = 4                  # full epochs to show per cell
SAMPLE_STRATEGY    = 'random'           # 'random' (default) or 'top_rate'
RANDOM_SEED        = None               # int for reproducible sampling; None = fresh
DPI                = 250                # 200–300 is good for visual inspection
OVERWRITE_QC_PNGS  = True               # re-render existing PNGs

sample_df, png_paths = ra.sample_and_plot_sorting_qc(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
    cell_types=CELL_TYPES,
    n_cells_per_type=N_CELLS_PER_TYPE,
    n_epochs=N_EPOCHS,
    sample_strategy=SAMPLE_STRATEGY,
    random_seed=RANDOM_SEED,
    dpi=DPI,
    overwrite=OVERWRITE_QC_PNGS,
)
print(f'\n→ wrote {len(png_paths)} PNG(s); open them with the system viewer.')


## 8. Interactive sorting-QC GUI (`ra.sorting_qc_gui`)

Notebook ipywidgets panel — same diagnostic as §7 but **one click =
one window**, on demand. Designed for remote-NAS sessions where
loading a full epoch (~1 GB at 20 kHz × 512-electrode 12-bit) is
wasteful.

### Controls (top to bottom)

| widget | what it does |
|---|---|
| **Cell** | dropdown over `QC-pass ∩ visual-QC good` for cell types `OnP`, `OnM` (extendable). Labels show cell type, protocol cell id, matched noise id, mean rate. |
| **Epoch** | which epoch of `EyeMovementTrajectoryAlternatingBackground` to load (~60 s each in this protocol). |
| **Electrode** | 1st / 2nd / 3rd top-amplitude electrode for the chosen cell's EI. Switching rank is **free** (`rt.data` already holds all 512 electrodes for the cached window). |
| **Window (s)** | slider + `start (s)` / `end (s)` text boxes (synced). Type for precise values; the estimated MB on the wire is shown next to the slider. |
| **Appearance** (accordion) | trace color / line width, spike color / marker size, HP cutoff (Hz), manual y-range, figure width. **All re-render from cache with zero I/O.** |
| **View** (row) | ⊕ zoom in · ⊖ zoom out · ◀ pan · pan ▶ — shrink/expand or shift the window by half its width. Zoom-in stays **inside the loaded data → free**; pan past the loaded window re-fetches. |
| **Load raw trace** | the main button that may trigger NAS bytes. Idempotent: re-clicking the same (epoch, window) is served from cache, and any sub-range of the loaded window also hits the cache. |
| **Overlay detected spikes** | toggle red dots + raster strip. Zero I/O. |
| **Bandwidth chip** | green = local mount (SSD, USB, …), amber = network mount (SMB / NFS / AFP). Ticks up only on real reads; reset button zeroes it. *Caveat*: macOS page-cache means this is an **upper bound** on wire bytes. |

### What re-fetches over the network vs what is free

| action | re-reads file? |
|---|---|
| First Load click for a new (cell, epoch, window) | **yes** |
| Same Load click again | no (idempotency guard) |
| Switch electrode rank | no |
| Toggle spike overlay | no |
| Any appearance widget (color, lw, HP cutoff, …) | no |
| ⊕ zoom in (or smaller window inside the loaded one) | no — served from cache |
| ⊖ zoom out / ◀ pan / pan ▶ past the loaded window | **yes** for the new bytes only |

### Vector output

Plots are **always vector SVG** — browser pinch / Ctrl-+ stays crisp
at any zoom level. For a *native* pan / zoom-rectangle toolbar inside
the figure (with a draggable selection rectangle), `pip install
ipympl` then add `%matplotlib widget` at the top of the notebook;
the GUI surfaces a one-line hint when ipympl is absent.

In [ ]:
from IPython.display import display

# Launches an ipywidgets panel for the pipeline built in §3.
# Pick cell → epoch → top-3 electrode → window (slider or
# FloatText) → 'Load raw trace'. Appearance accordion exposes
# trace/marker style and HP-cutoff frequency. The bandwidth
# meter at the bottom tracks cumulative MB read from disk.
display(ra.sorting_qc_gui(response_block))


## 9. Run the archive for one or many dates (standalone)

**Self-contained section** — run cell 1 (imports), then jump here.
`ra.analyze_experiments(dates, protocol_search=...)` packages every
step the earlier cells did manually into a single call per date:
`ss_version` + typing file + datafile auto-detect, pipeline build,
optional rig calibration, type normalization, QC, composite
`mosaic.png` (with temporal-filter + ISI rows), and per-cell
`cell_<id>_raster.png` + `cell_<id>_psth.png`.

### Defaults assume `EyeMovementTrajectoryAlternatingBackground`

`PROTOCOL_SEARCH = "AlternatingBackground"` substring-matches the full
Java class name. Change it to run the same archive over a different
protocol family — but condition coloring and QC thresholds may need
tuning for that protocol.

### Visual-QC integration

For any date that already has a `visual_qc.csv` in its archive folder,
the batch driver restricts that date's per-cell PNG step to cells
tagged `good`. Pass `respect_visual_qc=False` to override.

### Save toggle

`SAVE_FIGURES = True` ⇒ render and **overwrite** all PNGs.
`SAVE_FIGURES = False` ⇒ list the batch only and stop. The user
prefers this explicit gate over an auto-detect "does the PNG exist?"
check — don't re-introduce implicit skipping.

### Failure handling

`on_error="log"` so one bad date is recorded in the returned summary
DataFrame instead of aborting the batch. `n_jobs=-1` uses every CPU
core for per-cell PNG rendering.

In [ ]:
# Section 18 is SELF-CONTAINED — you only need cell 1 (imports) to run it.
# It will query the protocol registry on its own and dispatch the archive
# pipeline over every experiment found, in parallel.

import os
import pandas as pd

# ---- USER INPUT --------------------------------------------------------
# Yes/no: should we (re)save figures for every cell?
#   True  → render and OVERWRITE all PNGs (use this to refresh stale plots).
#           Per-date visual_qc.csv (if present) restricts the per-cell
#           PNG step to cells tagged 'good'.
#   False → skip the archive step entirely (just list the batch and stop).
SAVE_FIGURES = True
# ------------------------------------------------------------------------

PROTOCOL_SEARCH = 'AlternatingBackground'   # substring matched against protocol names

# Build the date list from the protocol registry, keeping only experiments
# whose sort output is present on disk.
_exp_search = ra.get_datasets_from_protocol_names(PROTOCOL_SEARCH)
_available = set(os.listdir(ra.ANALYSIS_DIR))
_exp_search = _exp_search[_exp_search['exp_name'].isin(_available)]
batch_dates = _exp_search['exp_name'].unique().tolist()

# Subset variants — uncomment / adapt as needed:
# batch_dates = ['20220823C', '20221123C', '20230502C']                                            # hand-pick
# batch_dates = _exp_search.query("exp_name >= '20230101C'")['exp_name'].unique().tolist()        # by date
# batch_dates = _exp_search.query('NDF == 2.0')['exp_name'].unique().tolist()                     # by NDF

print(f'Batch run over {len(batch_dates)} dates: {batch_dates}')
print(f'SAVE_FIGURES = {SAVE_FIGURES}  '
      f'({"overwrite all PNGs" if SAVE_FIGURES else "skip archive step"})')

if not SAVE_FIGURES:
    print('SAVE_FIGURES is False — not calling ra.analyze_experiments. '
          'Set SAVE_FIGURES = True above to (re)render PNGs.')
else:
    results = ra.analyze_experiments(
        batch_dates,
        protocol_search=PROTOCOL_SEARCH,    # resolves datafile per date
        fit_calibration=False,              # True on first pass to seed calibrations
        overwrite=True,                     # resave every PNG (driven by SAVE_FIGURES)
        n_jobs=-1,                          # all CPU cores for per-cell rendering
        on_error='log',                     # keep going past per-date failures
        respect_visual_qc=True,             # restrict to good-tagged cells when present
        verbose=True,
    )

    summary = pd.DataFrame([
        {k: r.get(k) for k in
         ['exp_name', 'datafile_name', 'chunk_name',
          'n_cells_total', 'n_cells_passed_qc', 'ndf', 'error']}
        for r in results
    ])
    display(summary)


## 10. Offline data store (`offline.h5`) — build once, reload fast

After §5/§6 leaves a curated visual-QC set, `ra.load_or_build_offline`
packages everything an analysis needs — metadata, condition table,
per-cell spike times, smoothed PSTHs, STA fit, EI summary — into a
single HDF5 at
`<OUTPUT_DIR>/<exp>/eye_movement_alt_bg/offline.h5`. Subsequent
sessions just **load** the file; no DataJoint, no SSD pipeline rebuild.

- **First call**: builds the pipeline → runs §4 QC → intersects with
  `visual_qc.csv` (good cells only) → writes `offline.h5`. ~1–2 min/date.
- **Re-runs**: `ra.load_offline_data(exp)` returns an `OfflineDataset`
  in <1 s. Pass `overwrite=True` to rebuild from source.
- **Cross-date**: `ra.load_offline_many()` → `{exp_name: OfflineDataset}`
  for every date that has `offline.h5`.

### `OfflineDataset` API

| attribute / method | what it is |
|---|---|
| `ds.meta`, `ds.timing` | scalars (exp id, datafile, ndf, preTime/stimTime/sample_rate) |
| `ds.epochs` | DataFrame, one row per epoch (`currentImageName`, `currentBackgroundScale`, …) |
| `ds.cells` | DataFrame, one row per saved cell (cell_type, STA fit, EI stats) |
| `ds.spike_times(cell_id)` | list of arrays (ms), one per epoch |
| `ds.psth_matrix(cell_id)` | `(n_epochs, n_bins)` Hz, Gaussian-smoothed |
| `ds.psth_time_ms()` | shared bin-center time axis |

### Why HDF5 and not Parquet/Pickle

Ragged per-epoch spike-time arrays + a regular `(n_epochs, n_bins)`
PSTH matrix coexist naturally in HDF5 groups. Reload is ~0.2 s and
the file is portable across machines.

In [ ]:
# §10 — Build / load the offline store for one experiment.
# First call: 1–2 min; subsequent calls: <1 s.

EXP = '20221123C'
PROTOCOL = 'eye_movement_alt_bg'

ds = ra.load_or_build_offline(
    EXP, protocol=PROTOCOL,
    protocol_search='AlternatingBackground',
    overwrite=False, verbose=True,
)
print(ds)
print('cell types:', ds.cell_types())
display(ds.epochs.head())
display(ds.cells.head())


## 11. Offline analyses (`retinanalysis.protocols.eye_movement_alt_bg`)

Each analysis takes the `OfflineDataset` from §10 and returns a
DataFrame (or dict for population metrics). Results are saved next to
`offline.h5` so cross-date pooling in §12 is a single `pd.concat`.

| function | what it computes | output |
|---|---|---|
| `analyze_offline` | per-(cell-type × condition) mean PSTHs | dict (in-memory) |
| `spike_distance_analysis` | Victor-Purpura distance over a 5-s window per trial. Reports within-condition mean (variability inside a condition) and across-condition mean; `d_diff = d_across - d_within_avg > 0` means the condition modulates the response. | DataFrame + `spike_distance.csv` |
| `movie_repeat_analysis` | splits `stimTime` into cycle-1 vs cycle-2 (15 s + 15 s, drop first second). Reports per (cell, condition): correlation, RMSE, mean-rate ratio (adaptation index), and optional per-trial VP distance. | DataFrame + `movie_repeat.csv` |
| `population_time_scale_metrics` | time-resolved population-vector divergence between the two `currentBackgroundScale` levels per cell type (Cohen's d, Euclidean / cosine distance, cumulative |Δrate|, per-bin Mann-Whitney AUC). | dict (in-memory) |

These defaults are **tuned for `EyeMovementTrajectoryAlternatingBackground`**:
- Movie cycle = 15 s (one full Eye-Movement trajectory loop).
- Condition keys = `(currentImageName, currentBackgroundScale)`.
- Population comparison axis = `currentBackgroundScale`.

For a different protocol, write a new analyzer module under
`retinanalysis/protocols/<protocol_name>/` and call it from a
protocol-specific notebook.

In [ ]:
# §11a — Average PSTH by (cell type × condition). Offline = no DJ needed.
from retinanalysis.protocols import eye_movement_alt_bg as ema

r = ema.analyze_offline(ds, minimum_n=3)
print(f'cell types: {r["cell_types"]}')
print(f'{len(r["conditions"])} conditions, {len(r["time_ms"])} time bins')

ema.plot_psth_by_condition(r, show_individual_cells=False)


In [ ]:
# §11b — Movie-repeat comparison: cycle 1 vs cycle 2 (15s each, drop first 1s).
# compute_vp=False keeps it fast (~30 s); enable for per-trial VP timing differences.
mr = ema.movie_repeat_analysis(
    ds, cycle_sec=15.0, drop_first_sec=1.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    compute_vp=False,
)
print(f'rows: {len(mr)}')
display(mr.groupby('cell_type')[['n_trials',
                                  'rate_cycle1_hz', 'rate_cycle2_hz',
                                  'rate_ratio',
                                  'corr_cycle12', 'rmse_cycle12_hz']]
          .median().round(3))

# Save to disk for cross-date pooling
ema.save_movie_repeat(mr, ds.exp_name)


In [ ]:
# §11c — Population time-scale metrics: two backgroundScale levels per cell type.
# Returns dict {cell_type: {cohens_d_mean, pop_euclid_dist, pop_cosine_dist, ...}}.
import matplotlib.pyplot as plt

pm = ema.population_time_scale_metrics(
    ds, primary_key='currentBackgroundScale',
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    smooth_ms=100.0, minimum_n=3,
)
t_ms = pm['time_ms']
pre = ds.timing['preTime_ms']; stim = ds.timing['stimTime_ms']

types = pm['cell_types']
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ct in types:
    d = pm[ct]
    axes[0].plot(t_ms, d['cohens_d_abs_mean'], label=f'{ct} (n={d["n_cells"]})')
    axes[1].plot(t_ms, d['pop_euclid_dist'], label=ct)
    axes[2].plot(t_ms, d['cum_abs_divergence'], label=ct)
for ax in axes:
    ax.axvline(pre, color='r', lw=0.5, ls='--', alpha=0.5)
    ax.axvline(pre+stim, color='r', lw=0.5, ls='--', alpha=0.5)
axes[0].set_ylabel('mean |Cohen\'s d|')
axes[1].set_ylabel('pop Euclid dist')
axes[2].set_ylabel('cum |Δrate|·dt (spikes·cells)')
axes[2].set_xlabel('time (ms)')
axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle(f'{ds.exp_name}: low vs high background scale, time-resolved')
fig.tight_layout()


In [ ]:
# §11d — Spike-distance (Victor-Purpura) across backgroundScale, within image.
# Default pair_within=('currentImageName',): for each image, pair trials across
# the low vs high backgroundScale.  C-accelerated DP makes this fast — under 5s
# for ~170 cells × 5 images.
sd = ema.spike_distance_analysis(
    ds, window_sec=5.0, cost_per_sec=4.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    pair_within=('currentImageName',),   # hold image constant, compare BG scale
    n_trials_cap=None,                    # use every trial — fast in C
)
print(f'rows: {len(sd)}; cells: {sd["cell_id"].nunique()}; '
      f'images: {sd["group_key"].nunique()}')

display(sd.groupby('cell_type')[['d_within_avg', 'd_across', 'd_diff']]
          .agg(['median', 'count']).round(2))

ema.save_spike_distance(sd, ds.exp_name)


## 12. Cross-date aggregation

Once every experiment has been through §10–§11 (each writes
`offline.h5`, `spike_distance.csv`, `movie_repeat.csv` to its own
folder), pooling across dates is just a `concat`.

| call | returns |
|---|---|
| `ra.load_offline_many()` | `{exp_name: OfflineDataset}` for every experiment with `offline.h5` on disk |
| `ema.aggregate_psth_across_dates(offlines)` | pooled per-cell mean PSTHs as one `(n_cells_total, n_bins)` matrix per `(cell_type, condition)` — pass to `ema.plot_psth_by_condition` |
| `ema.load_spike_distance_many()` | long-format DataFrame, `exp_name`-tagged |
| `ema.load_movie_repeat_many()` | long-format DataFrame, `exp_name`-tagged |

Adding a new date to the pool: run §10 (and §11 if you want the
spike-distance / movie-repeat CSVs) for that date, then re-run §12
unchanged — `load_*_many()` picks it up automatically.

In [ ]:
# §12 — Cross-date pooled analyses.
offlines = ra.load_offline_many()  # all dates with offline.h5
print(f'experiments loaded: {len(offlines)}')
for exp, ds_ in offlines.items():
    print(f'  {exp}: {len(ds_.cell_ids)} cells, types={ds_.cell_types()}')

# Pool PSTHs across dates
pooled = ema.aggregate_psth_across_dates(offlines, minimum_n=5)
print(f'\npooled types: {pooled["cell_types"]} from {pooled["n_dates"]} dates')
ema.plot_psth_by_condition(pooled, show_individual_cells=False)

# Pool spike-distance & movie-repeat CSVs
sd_all = ema.load_spike_distance_many()
mr_all = ema.load_movie_repeat_many()
print(f'\nspike_distance rows: {len(sd_all)} from '
      f'{sd_all["exp_name"].nunique() if not sd_all.empty else 0} dates')
print(f'movie_repeat rows: {len(mr_all)} from '
      f'{mr_all["exp_name"].nunique() if not mr_all.empty else 0} dates')

# Headline cross-date summaries
if not sd_all.empty:
    display(sd_all.groupby(['cell_type'])
                  [['d_within_avg', 'd_across', 'd_diff']]
                  .agg(['median', 'count']).round(2))
if not mr_all.empty:
    display(mr_all.groupby(['cell_type'])
                  [['rate_ratio', 'corr_cycle12', 'rmse_cycle12_hz']]
                  .agg(['median', 'count']).round(3))
